# VGGFace2: image exploration

We use the official test archive as a **development pilot**. People used here must not be treated as untouched identities in a later final evaluation.

This notebook counts all local images, then decodes a reproducible random sample of 1,000 images. It does not train a model, create splits, or modify source images. Sample findings do not establish the quality of the full dataset.


## Recorded pilot findings

The first audit used seed 42 and a random sample of 1,000 images:

| Check | Result |
|---|---|
| Images on disk | 169,396 |
| Identities | 500 |
| Missing / unexpected paths against official list | 0 / 0 |
| Images per identity: min / median / max | 98 / 325 / 761 |
| Readable sampled images | 1,000 / 1,000 |
| Original sample modes | All RGB |
| Sample images with a side below 128 pixels | 221 (22.1%) |
| Exact pixel duplicate pairs within sample | 0 |

Manual review of the generated grids shows changes in pose, lighting and crop size. Some images include another face. These observations apply to the displayed examples, not every image. No exact duplicates in the sample does not rule out duplicates across the full archive.

Before training: review crops and resizing versus padding; select a fixed identity subset; check duplicates within that subset; and create shared splits for both models. Do not discard low-resolution images automatically based only on this sample.


In [ ]:
from pathlib import Path
import os
import json
from PIL import Image
from IPython.display import display

# Run from either the repository root or notebooks/.
REPO = Path.cwd()
if not (REPO / "src").is_dir():
    REPO = REPO.parent
assert (REPO / "src").is_dir(), "Open this notebook from the project directory"
os.chdir(REPO)
OUTPUT = REPO / "outputs/image_exploration"


## 1. Inventory and completeness

First run the image audit. The expected list must match the chosen archive. Wait for extraction to finish before using final counts. Download filename lists with `python -m src.data.explore` if needed.


In [ ]:
from src.data.explore_images import explore
report = explore(sample_size=1000, seed=42)
print("Completeness:", report["completeness"])


## 2. Images per identity

Look for classes with few images and large differences between classes. Class imbalance means some people appear more often than others. Both models must use the same selected identities and splits.


In [ ]:
print(json.dumps(report["images_per_identity"], indent=2))
display(Image.open(OUTPUT / "identity_counts.png"))


## 3. Inspect faces

Check whether faces are visible, whether crops are useful, and how pose, lighting, background and image quality vary. The first grid is random. In the second grid, each row shows four images assigned to the same person. Review suspicious labels manually.


In [ ]:
display(Image.open(OUTPUT / "random_faces.png"))
display(Image.open(OUTPUT / "same_identity_faces.png"))


## 4. Dimensions and brightness

Small images may lose detail when enlarged to 128 x 128. Different image shapes also mean that direct square resizing can distort faces. Compare resizing and padding before choosing a shared policy. Brightness is only a rough image statistic; it is not a measure of identity or skin tone.


In [ ]:
print("Width:", report["sample_width"])
print("Height:", report["sample_height"])
print("Sample images with a side below 128 pixels:", report["sample_smaller_than_128"])
for name in ("widths.png", "heights.png", "brightness.png"):
    display(Image.open(OUTPUT / name))


## 5. Readability and duplicates

These checks apply only to the sampled images. Exact duplicates are found by hashing decoded RGB pixels and dimensions after EXIF orientation correction. A clean sample does not prove that the whole archive has no corrupt files or duplicates. Near duplicates require a separate check before splitting.


In [ ]:
print("Sample checked:", report["sample_checked"])
print("Original image modes:", report["sample_original_modes"])
print("Unreadable sampled images:", report["sample_errors"])
print("Exact duplicate pairs in sample:", report["sample_exact_pixel_duplicate_pairs"])


## 6. Decisions before training

Record observations after running the notebook:

- Is extraction complete and does the inventory match the filename list?
- How many identities and images per identity will we keep?
- Do sampled faces show enough variation? Are there suspicious labels?
- Should we pad images instead of stretching them to a square?
- Which files need further review before the split?

Next: select a reproducible subset, audit duplicates within it, create train/validation/test splits, and build the DataLoaders. Keep all generated figures and identifying metadata local under `outputs/`. Clear notebook outputs before committing it.
